## 1. Imports
Everything the notebook needs, in one place (no repeated/duplicate import cells).

In [25]:
import json
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher, get_close_matches

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading


## 2. Define data sources (paths listed once, files read exactly once below)

In [26]:
files = [
    r"C:\Users\HP\Downloads\files (18)\neu_phone_directory.json",
    r"C:\Users\HP\Downloads\files (18)\neu_admissions_academic.json",
    r"C:\Users\HP\Downloads\files (18)\neu_campus_facilities.json",
    r"C:\Users\HP\Downloads\files (18)\neu_faculties_departments.json",
    r"C:\Users\HP\Downloads\files (18)\neu_general_info.json",
    r"C:\Users\HP\Downloads\files (18)\neu_hospital_departments.json",
    r"C:\Users\HP\Downloads\files (18)\neu_locations.json",
    r"C:\Users\HP\Downloads\files (18)\neu_qa_dataset.txt",
    r"C:\Users\HP\Downloads\files (18)\neu_research.json",
    r"C:\Users\HP\Downloads\files (18)\README.md",
    r"C:\Users\HP\Downloads\files (18)\level1111.csv"
]

## 3. Load every file ONCE
Builds the `Document` objects for embedding, and caches every parsed JSON file in `RAW_JSON_CACHE` so the doublecheck step further down can reuse it instead of reading the files from disk a second time.

In [27]:
import unicodedata

all_docs = []
ROOM_LOOKUP = {}      # normalized room_number -> room chunk text (exact lookup for short codes like "U08")
LECTURER_INDEX = []   # list of (name_tokens: frozenset, room_text: str, display_name: str)

# Academic titles/degree abbreviations to drop when comparing names, so "Prof. Dr. X Y"
# and a question like "where is X Y" both reduce to the same meaningful token set.
_TITLE_WORDS = {
    "prof", "dr", "assoc", "asst", "eng", "msc", "mse", "ma", "med", "phd",
    "doc", "aras", "gor", "dt", "mr", "mrs", "ms",
}

def _fold_diacritics(s: str) -> str:
    """ASCII-fold a string so Turkish characters compare equal regardless of how they're
    typed: 'Akgün' / 'akgun' / 'AKGUN' all fold to 'akgun'. Turkish dotted/dotless I
    (İ/ı) aren't handled correctly by plain NFKD+lower, so they're special-cased first."""
    s = s.replace("İ", "I").replace("ı", "i")
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower()

def _name_tokens(raw: str) -> frozenset:
    """Extract meaningful, diacritic-folded name tokens from a lecturer string or a
    question, dropping punctuation and academic titles/degrees."""
    cleaned = re.sub(r"[.,]", " ", str(raw))
    words = [_fold_diacritics(w) for w in cleaned.split()]
    return frozenset(w for w in words if w and w not in _TITLE_WORDS)

# The level_u.csv room_number column stores values like "Room U08" / "Room 6"
# (the word "Room" is baked into the value itself, not just the room's name).
_ROOM_PREFIX_RE = re.compile(
    r"^(?:room|office|lab|hall|classroom)\.?\s*(?:no\.?|number)?\s*[:\-]?\s*",
    re.IGNORECASE,
)

def _normalize_room_key(val) -> str:
    """Normalize a room_number value (from the CSV OR extracted from a user's question)
    into the SAME lookup key, regardless of which side it came from. Handles:
      1. If ANY row in the room_number column is blank, pandas silently upcasts the
         WHOLE column to float64, turning e.g. 6 into 6.0 -> "6.0". We collapse that
         back to "6" so purely-numeric rooms ("Room 6") still match.
      2. A leading "Room"/"Office"/"Lab"/etc. word, since the CSV's room_number values
         already include it (e.g. "Room U08"), but questions get parsed with that word
         stripped off separately -- so both sides need the SAME stripping to line up.
      3. Spaces/dashes/case differences ("U-08" vs "u08" vs "U 08") are stripped so
         they all resolve to the same key.
    """
    s = str(val).strip().lower()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    s = _ROOM_PREFIX_RE.sub("", s)
    return s.replace("-", "").replace(" ", "")

# Function to load JSON files
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Function to load text files
def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def parse_qa_dataset(text: str):
    """Parses the `Q: ... / A: ...` pairs in neu_qa_dataset.txt into a list of
    dicts. Pure text parsing - no LLM, no embedding call here."""
    lines = [l for l in text.split("\n") if not l.strip().startswith("#")]
    text = "\n".join(lines)
    blocks = re.split(r"\n(?=Q:\s)", text)
    pairs = []
    for b in blocks:
        b = b.strip()
        if not b.startswith("Q:"):
            continue
        m = re.match(r"Q:\s*(.*?)\nA:\s*(.*)", b, re.DOTALL)
        if m:
            pairs.append({"question": m.group(1).strip(), "answer": m.group(2).strip()})
    return pairs

for file in files:

    # ==========================
    # JSON FILES
    # ==========================
    if file.endswith(".json"):
        data = load_json(file)

        # Faculties dataset
        if isinstance(data, dict) and "faculties" in data:

            for faculty in data["faculties"]:

                departments = "\n".join(
                    f"- {d}" for d in faculty.get("departments", [])
                )

                degree_types = ", ".join(
                    faculty.get("degree_types", [])
                )

                text = f"""
This document is about the Faculty of {faculty['name']}.

Faculty Name: {faculty['name']}

Departments:
{departments}

Degree Types:
{degree_types}

Website:
{faculty.get('website','')}
"""

                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "faculty_name": faculty["name"],
                            "type": "faculty"
                        }
                    )
                )

        # Hospital departments dataset -> one readable document per department
        elif isinstance(data, dict) and "departments" in data and isinstance(data["departments"], dict):

            for dept_id, d in data["departments"].items():

                hours_parts = [
                 f"{k.replace('_', '–').title()}: {v}"
                 for k, v in d.get("hours", {}).items()
                ]
                hours_text = "; ".join(hours_parts) if hours_parts else "Not specified"

                text = f"""
This document describes a department at NEU Hospital.

Department: {d.get('name_en', dept_id)}
Floor: {d.get('floor', '')}
Block: {d.get('block', '')}
Hours: {hours_text}
Facilities: {', '.join(d.get('facilities', []))}
Notes: {d.get('notes', '')}
"""

                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "department_name": d.get("name_en", dept_id),
                            "type": "hospital_department"
                        }
                    )
                )

        # Campus locations dataset -> one readable document per building/location
        elif isinstance(data, dict) and ("academic_buildings" in data or "key_locations" in data):

            entries = list(data.get("academic_buildings", [])) + list(data.get("key_locations", []))

            for loc in entries:

                text_parts = [
                    "This document describes a location on the NEU campus.",
                    f"Name: {loc.get('name', '')}",
                    f"Description: {loc.get('location_description', '')}",
                    f"Directions: {loc.get('directions', '')}",
                ]
                if loc.get("hours"):
                    text_parts.append(f"Hours: {loc['hours']}")
                phone = loc.get("phone") or loc.get("phone_extension")
                if phone:
                    text_parts.append(f"Phone: {phone}")
                if loc.get("departments_housed"):
                    text_parts.append("Departments housed here: " + ", ".join(loc["departments_housed"]))

                all_docs.append(
                    Document(
                        page_content="\n".join(text_parts),
                        metadata={
                            "source": file,
                            "location_name": loc.get("name", ""),
                            "type": "location"
                        }
                    )
                )

        # Every other JSON file (admissions, general_info, campus_facilities, research, ...):
        # flatten into (readable label -> verbatim value) leaf pairs instead of dumping the
        # whole file as one raw JSON blob. Each leaf becomes its own small, readable document,
        # so a doublecheck hit returns a clean sentence instead of a jagged JSON fragment.
        else:

            def _flatten_json(obj, prefix=""):
                items = []
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        label = f"{prefix} {k.replace('_', ' ')}".strip()
                        if isinstance(v, dict):
                            items.extend(_flatten_json(v, label))
                        elif isinstance(v, list):
                            if all(isinstance(e, (str, int, float)) for e in v):
                                items.append((label, "; ".join(str(e) for e in v)))
                            else:
                                for entry in v:
                                    sub_prefix = label
                                    if isinstance(entry, dict):
                                        name = entry.get("name") or entry.get("title")
                                        if name:
                                            sub_prefix = f"{label} {name}"
                                    items.extend(_flatten_json(entry, sub_prefix))
                        else:
                            items.append((label, str(v)))
                return items

            for label, value in _flatten_json(data):
                all_docs.append(
                    Document(
                        page_content=f"{label}: {value}",
                        metadata={
                            "source": file,
                            "type": "field"
                        }
                    )
                )

    # ==========================
    # CSV FILES
    # ==========================
    elif file.endswith(".csv"):

         df = pd.read_csv(file, encoding="cp1252")

         # Group all rows that belong to the same room
         grouped = df.groupby("room_number")

         def _clean(v):
             """Turn missing/NaN/blank CSV cells into None instead of the literal
             string "nan" (which pandas produces when an empty cell is f-string'd
             directly), so blank fields can be skipped instead of printed."""
             if v is None:
                 return None
             if isinstance(v, float) and pd.isna(v):
                 return None
             s = str(v).strip()
             if not s or s.lower() == "nan":
                 return None
             return s

         for room_number, group in grouped:

           # One clean "Name (Position)" line per lecturer -- skip rows with no name,
           # and skip the position entirely if it's missing rather than printing "nan".
           lecturers = []
           for _, row in group.iterrows():
               name = _clean(row.get("Lecturers"))
               if not name:
                   continue
               pos = _clean(row.get("Position/Title"))
               lecturers.append(f"{name} ({pos})" if pos else name)

           building        = _clean(group.iloc[0].get("building"))
           floor           = _clean(group.iloc[0].get("floor"))
           room_name       = _clean(group.iloc[0].get("room_name"))
           room_type       = _clean(group.iloc[0].get("room_type"))
           location_desc   = _clean(group.iloc[0].get("location_description"))

           # Build the text as a list of lines and only include fields that actually
           # have a value -- no more "location description: nan" / blank trailing lines.
           lines = ["This document describes a room at the NEU campus.", ""]
           if building:      lines.append(f"Building: {building}")
           if floor:         lines.append(f"Floor: {floor}")
           lines.append(f"Room Number: {room_number}")
           if room_name:     lines.append(f"Room Name: {room_name}")
           if room_type:     lines.append(f"Room Type: {room_type}")
           if location_desc: lines.append(f"Location description: {location_desc}")
           if lecturers:
               lines.append("Lecturers:")
               lines.extend(f"- {l}" for l in lecturers)
           text = "\n".join(lines)
           # `text` above stays structured -- it's what gets embedded into FAISS, and a
           # structured layout is fine (even good) for that. But it's a bad *chat* answer.
           # Build a natural-language sentence version, used only for the exact-match
           # shortcuts below, so what the user actually reads looks like a reply, not a
           # database dump.
           room_label = str(room_number)
           if room_name and room_name.strip().lower() != room_label.lower():
               room_label += f' ("{room_name}")'

           where_bits = []
           if building: where_bits.append(f"the {building.title()}")
           if floor:    where_bits.append(floor)
           where_clause = " in " + ", ".join(where_bits) if where_bits else ""

           room_sentence = f"{room_label} is located{where_clause}."
           if lecturers:
               if len(lecturers) == 1:
                   room_sentence += f" {lecturers[0]} is based there."
               else:
                   room_sentence += " The people based there are: " + ", ".join(lecturers) + "."

           all_docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file,
                        "type": "room",
                        "building": building or "",
                        "floor": floor or "",
                        "room_number": str(room_number),
                        "room_name": room_name or "",
                        "room_type": room_type or "",
                        "location description": location_desc or "",
                        "lecturers": lecturers,
                       }
                )
            )

           # Index this room under a normalized key so exact codes -- whether
           # alphanumeric ("U08") or purely numeric ("6") -- can be looked up directly,
           # bypassing semantic embedding search (unreliable for short exact codes).
           room_key = _normalize_room_key(room_number)
           if room_key:
               ROOM_LOOKUP[room_key] = room_sentence

           # Index each lecturer in this room by diacritic-folded name tokens, so a
           # question like "where is Yigit Akgün" can be matched directly against the
           # CSV data -- this bypasses both the semantic embedding search AND the
           # ASCII-only tokenizer used later for the confidence check, neither of which
           # handles Turkish characters (ü/ğ/ş/ı/ö/ç) correctly.
           for _, row in group.iterrows():
               lect_raw = row.get("Lecturers", "")
               if pd.isna(lect_raw) or not str(lect_raw).strip():
                   continue
               name_tokens = _name_tokens(lect_raw)
               if len(name_tokens) < 2:  # need at least first+last name to be identifiable
                   continue

               display_name = str(lect_raw).strip()
               pos = _clean(row.get("Position/Title"))
               person_lead = f"{display_name} ({pos})" if pos else display_name

               colleagues = [l for l in lecturers if not l.startswith(display_name)]
               person_sentence = f"{person_lead} is based in {room_label}, located{where_clause}."
               if colleagues:
                   person_sentence += " Also based there: " + ", ".join(colleagues) + "."

               LECTURER_INDEX.append((name_tokens, person_sentence, display_name))

    # ==========================
    # TXT / MD FILES
    # ==========================
    elif file.endswith(".txt") or file.endswith(".md"):

        text = load_txt(file)

        # The curated Q&A dataset gets folded in as ONE Document per (question,
        # answer) pair -- embedding just the question -- so it lands in the
        # SAME FAISS store as everything else and can be retrieved through the
        # SAME top-K retriever, instead of being embedded a second time later
        # outside FAISS.
        if "qa_dataset" in file.lower():
            qa_pairs = parse_qa_dataset(text)
            for qp in qa_pairs:
                all_docs.append(
                    Document(
                        page_content=qp["question"],
                        metadata={
                            "source": file,
                            "type": "qa_pair",
                            "answer": qp["answer"],
                        }
                    )
                )
        else:
            all_docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file,
                        "type": "text"
                    }
                )
            )

    else:
        print(f"[skip] Unsupported file type: {file}")

print("Total documents (incl. QA pairs):", len(all_docs))


Total documents (incl. QA pairs): 1394


In [28]:
# --- DIAGNOSTIC: inspect the CSV room-matching dataset ---------------------------
# Run this AFTER the "Load every file ONCE" cell (and after restarting the kernel /
# re-running the notebook top-to-bottom, so ROOM_LOOKUP reflects the latest code).

print("Total rooms indexed in ROOM_LOOKUP:", len(ROOM_LOOKUP))
print("\nFirst 20 keys in ROOM_LOOKUP:")
for k in list(ROOM_LOOKUP.keys())[:20]:
    print(" -", repr(k))

print("\nIs 'u08' present as a key?", "u08" in ROOM_LOOKUP)

# Show the raw room_number column straight from level_u.csv, so you can see exactly
# what values/dtype pandas actually parsed (this is the #1 cause of mismatches --
# e.g. dtype, stray whitespace, different capitalization, or a different column name).
_csv_path = [f for f in files if f.lower().endswith("level_u.csv")]
if _csv_path:
    _df_check = pd.read_csv(_csv_path[0], encoding="cp1252")
    print("\nColumns in level_u.csv:", list(_df_check.columns))
    if "room_number" in _df_check.columns:
        print("room_number dtype:", _df_check["room_number"].dtype)
        print("Unique room_number values (first 30):")
        print(_df_check["room_number"].unique()[:30])
    else:
        print("\n[!] 'room_number' column NOT found in level_u.csv -- this is the bug.")
        print("    The CSV-loading code expects a column literally named 'room_number'.")
else:
    print("level_u.csv not found in `files` list.")


Total rooms indexed in ROOM_LOOKUP: 118

First 20 keys in ROOM_LOOKUP:
 - '10'
 - '101'
 - '102'
 - '103'
 - '104'
 - '105'
 - '106'
 - '107'
 - '108'
 - '109'
 - '11'
 - '110'
 - '111'
 - '112'
 - '113'
 - '114'
 - '115'
 - '117'
 - '118'
 - '119'

Is 'u08' present as a key? True
level_u.csv not found in `files` list.


## 4. Embedding model

In [29]:
embeddings = HuggingFaceEmbeddings( # convert text into numbers
    model_name="sentence-transformers/all-MiniLM-L6-v2" # Vector Database Implementation
)

## 5. Chunking

In [30]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))

Total chunks: 1436


## 6. Vectorize — build the FAISS vector database (once)

In [31]:
vectorstore = FAISS.from_documents(chunks, embeddings) # chunks: These are your small pieces of text after splitting, embeddings: It converts text → numbers (vectors).

## 7. Retriever (connects to the vector database)

In [32]:
retriever = vectorstore.as_retriever( # retriever : Finds and returns the most relevant pieces of information from your knowledge base based on a user question.
    search_type="similarity",         # This connects your chatbot to the FAISS database.
    search_kwargs={"k": 100} 
)

## 8. Prompt template for RAG

In [33]:
SYSTEM_PROMPT = """\
You are NEU Assistant, a friendly and helpful AI for Near East University (NEU) in Nicosia, Cyprus.

Your personality:
- Friendly, helpful, and professional
- Respond concisely but completely

Your responsibilities:
1. ACADEMIC       → Answer questions about faculties, departments, programmes, admissions, and courses.
2. LOCATION       → Provide building names, campus map references, or directions when asked where something is.
3. GENERAL INFO   → Cover research centres, hospital, facilities, events, and university history.
4. ANY QUESTION   → If the question is not about NEU, answer it naturally and helpfully from your own general knowledge. Do NOT say you can't help or redirect to NEU.
"""

RAG_TEMPLATE = """\
{system_prompt}

--- Relevant Context from NEU Knowledge Base ---
{context}
------------------------------------------------

User question: {question}

Answer directly (no greeting, cite your source at the end):"""

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    partial_variables={"system_prompt": SYSTEM_PROMPT},
    template=RAG_TEMPLATE,
)

print("Prompt template ready. ✅")

Prompt template ready. ✅


## 9. LLM

In [34]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("Local Ollama LLM loaded. ✅ (qa_chain is built later, after rag_prompt is defined)")

Local Ollama LLM loaded. ✅ (qa_chain is built later, after rag_prompt is defined)


## 10. Connect the vector database to the LLM (the RAG chain)

In [35]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("QA chain ready.")

QA chain ready.


## 11. Doublecheck — is the top retrieved chunk good enough to answer with, no LLM?

Matches the architecture exactly: the SAME `vectorstore` and top-K retrieval used for the LLM path is reused here. `dataset_answer()` takes the best-scoring retrieved chunk and asks one question — is it similar enough to the user's question to trust as-is?

- **YES** -> return that chunk's own text (its stored `answer` if it's a curated Q&A chunk, otherwise the chunk's own content) as the final answer. Zero LLM calls.
- **NO** -> return `None`, and the caller falls back to `qa_chain` (retrieved chunks + question -> LLM) further up.

No separate raw-JSON/`difflib` lookups anymore — one retrieval, one confidence check.

In [36]:
# Words too generic/common to count as a meaningful match on their own.
_STOPWORDS = {"of", "the", "and", "in", "at", "is", "are", "for", "a", "an", "to", "on", "s",
              "what", "who", "why", "how", "when", "does", "do", "can", "i", "you", "me",
              "it", "this", "that", "be", "about"}

def _tokenize(s: str):
    # Modified: Keeps words if they are >= 3 characters OR if they contain a digit (e.g., "21", "b3")
    return [
        w for w in re.findall(r"[a-z0-9]+", s.lower()) 
        if w not in _STOPWORDS and (len(w) >= 3 or any(char.isdigit() for char in w))
    ]

# Tune: higher = stricter (fewer, safer, no-LLM hits).
CHUNK_MATCH_THRESHOLD = 0.72

def _clean_label(doc) -> str:
    label = (
        doc.metadata.get("faculty_name")
        or doc.metadata.get("department_name")
        or doc.metadata.get("location_name")
        or doc.metadata.get("source", "NEU Knowledge Base")
    )
    if "\\" in label or "/" in label:
        label = label.replace("\\", "/").split("/")[-1]
    return label

def _cos_sim_from_l2(l2_dist: float) -> float:
    # embeddings are unit-normalized -> cosine_sim = 1 - (l2_distance^2)/2
    return 1 - (l2_dist ** 2) / 2

def dataset_answer(question: str, k: int = 5):
    """
    THE doublecheck: retrieve the top-K chunks from the SAME vectorstore/retriever
    used for the LLM path, and check if the best one is similar enough to trust
    with zero LLM calls.
    """
    # --- Exact room-code shortcut ---------------------------------------------------
    # Semantic embeddings often fail on short exact codes (e.g. "Room U08", "Room 6"),
    # since a model like MiniLM has little signal to distinguish "U08" from "U09", or
    # a bare room number from any other number in the question. Check the CSV's own
    # room_number index for an exact match first; only fall through to the
    # semantic/embedding path below if nothing matches exactly.
    #   1) context-aware: "room/office/lab 6", "room no. U08" -> pulls out just the code
    #   2) generic fallback: a bare alphanumeric code with no context word, e.g. "U08"
    _ROOM_CONTEXT_RE = re.compile(
        r"\b(?:room|office|lab|hall|classroom)\.?\s*(?:no\.?|number)?\s*[:\-]?\s*([a-z]{0,3}-?\d{1,4}[a-z]?)\b",
        re.IGNORECASE,
    )
    candidate_codes = [m.group(1) for m in _ROOM_CONTEXT_RE.finditer(question)]
    candidate_codes += re.findall(r"[a-z]{1,3}-?\d{1,4}[a-z]?", question.lower())

    for raw_code in candidate_codes:
        code_key = _normalize_room_key(raw_code)
        if code_key in ROOM_LOOKUP:
            return {
                "answer": ROOM_LOOKUP[code_key],
                "source": "level_u.csv",
                "method": f"exact-room-match ({raw_code})",
            }

    # --- Exact lecturer-name shortcut -------------------------------------------
    # Same underlying problem as room codes, but worse for names: semantic embeddings
    # AND the ASCII-only _tokenize() below both mishandle Turkish characters (ü/ğ/ş/ı/
    # ö/ç), so a person's name can fail to match even when it's right there in a
    # room's Lecturers list. Match diacritic-folded name tokens against the CSV's own
    # lecturer index directly, before doing anything semantic.
    q_name_tokens = _name_tokens(question)
    if q_name_tokens:
        best_score = 0
        matches_at_best = []  # (room_text, display_name) pairs tied for the best score
        for name_tokens, room_text, display_name in LECTURER_INDEX:
            if name_tokens.issubset(q_name_tokens):
                score = len(name_tokens)
                if score > best_score:
                    best_score, matches_at_best = score, [(room_text, display_name)]
                elif score == best_score:
                    matches_at_best.append((room_text, display_name))

        # Require >=2 matched name tokens (first+last), so a single common word can't
        # misfire on its own.
        if matches_at_best and best_score >= 2:
            # De-duplicate identical room chunks first: the same person can appear as
            # more than one CSV row for the SAME room (near-duplicate formatting), which
            # isn't a real ambiguity.
            distinct_rooms = {}
            for room_text, display_name in matches_at_best:
                distinct_rooms.setdefault(room_text, display_name)

            if len(distinct_rooms) == 1:
                room_text, display_name = next(iter(distinct_rooms.items()))
                return {
                    "answer": room_text,
                    "source": "level_u.csv",
                    "method": f"exact-lecturer-match ({display_name})",
                }
            else:
                # The person genuinely appears in more than one room (e.g. two roles) --
                # list all of them rather than guessing or silently giving up.
                combined = "\n\n---\n\n".join(distinct_rooms.keys())
                return {
                    "answer": f"This name is associated with more than one location:\n\n{combined}",
                    "source": "level_u.csv",
                    "method": "exact-lecturer-match (multiple rooms)",
                }

    hits = vectorstore.similarity_search_with_score(question, k=k)
    if not hits:
        return None

    q_tokens = set(_tokenize(question))

    for doc, l2_dist in hits:
        cos_sim = _cos_sim_from_l2(l2_dist)
        if cos_sim < CHUNK_MATCH_THRESHOLD:
            break  # hits are sorted best-first, nothing further will pass either

        # Require a real shared keyword too
        compare_text = doc.metadata.get("question", doc.page_content)
        if not (q_tokens & set(_tokenize(compare_text))):
            continue

        is_qa_pair = doc.metadata.get("type") == "qa_pair"
        answer_text = doc.metadata["answer"] if is_qa_pair else doc.page_content.strip()

        return {
            "answer": answer_text,
            "source": _clean_label(doc),
            "method": f"dataset-match (chunk, score={cos_sim:.2f})",
        }

    return None  # nothing confident enough -> caller falls back to the LLM

## 12. Spell correction / normalization

In [37]:
NEU_VOCAB = [
    "hello", "hi", "hey", "greetings",
    "department", "faculty", "programme", "program", "course",
    "degree", "admission", "tuition", "scholarship", "enrol",
    "register", "bachelor", "master", "phd", "gpa",
    "where", "location", "located", "building", "floor",
    "room", "map", "campus", "directions", "find",
    "research", "hospital", "facilities", "library",
    "engineering", "dentistry", "medicine", "science",
    "arts", "business", "law", "pharmacy", "architecture",
]

# Common English words that must NEVER be auto-"corrected" into NEU_VOCAB terms.
# (Fixes bug: "there" was being matched to "where" at cutoff=0.8, turning
#  greetings like "Hi there!" into LOCATION-intent queries.)
SAFE_WORDS = {
    "there", "here", "the", "is", "are", "a", "an", "in", "on", "at",
    "what", "who", "why", "how", "when", "this", "that", "it", "to",
    "for", "of", "and", "or", "do", "does", "can", "i", "you", "me",
}

def correct_spelling(text: str) -> str:
    words = text.split()
    corrected = []
    for word in words:
        clean = word.strip("?!.,").lower()

        # Skip correction entirely for known safe/common words and very short words
        # (short words produce too many spurious fuzzy matches).
        if clean in SAFE_WORDS or len(clean) <= 3:
            corrected.append(word)
            continue

        matches = get_close_matches(clean, NEU_VOCAB, n=1, cutoff=0.85)
        if matches and matches[0] != clean:
            print(f"  [spell-check] '{word}' → '{matches[0]}'")
            corrected.append(matches[0])
        else:
            corrected.append(word)
    return " ".join(corrected)

## 13. Rule-based intent classification

In [38]:
GREETING_PATTERNS = re.compile(
    r"^\s*(hi|hello|hey|greetings|good\s*(morning|afternoon|evening)|howdy|salut|merhaba|selam|"
    r"how are you|how are u|how r u|what's up|sup|naber|nas\u0131ls\u0131n)(\s+\w+)?\W*$",
    re.IGNORECASE # ignore upper and lower case
)

# Catches bot-directed questions like "where are you", "what are you", "who are you"
BOT_DIRECTED = re.compile(
    r"^\s*(where|what|who)\s+are\s+you\W*$",
    re.IGNORECASE
)

LOCATION_KEYWORDS = re.compile(
    r"\b(where|location|located|building|floor|room|map|campus|directions?|find)\b",
    re.IGNORECASE
)

ACADEMIC_KEYWORDS = re.compile(
    r"\b(departments?|facult(?:y|ies)|programmes?|programs?|courses?|degrees?|admissions?|tuition|scholarships?|enrol(?:l(?:ed|ment)?)?|register(?:ed)?|bachelors?|masters?|phd|gpa)\b",
    re.IGNORECASE
)

def detect_intent(text: str) -> str:
    """Return one of: 'greeting' | 'bot' | 'location' | 'academic' | 'general'"""
    if GREETING_PATTERNS.match(text): # match : checks only the start of the text
        return "greeting"
    if BOT_DIRECTED.match(text):
        return "bot"
    if LOCATION_KEYWORDS.search(text): # search: checks whole of text
        return "location"
    if ACADEMIC_KEYWORDS.search(text):
        return "academic"
    return "general"

# Smoke-test
samples = [
    ("Hello!",                              "greeting"),
    ("Where are you?",                      "bot"),
    ("Where is the Faculty of Engineering?","location"),
    ("What departments does NEU have?",     "academic"),
    ("Tell me about NEU research centres.", "general"),
]
for text, expected in samples:
    pred   = detect_intent(text)
    status = "✅" if pred == expected else "❌"
    print(f"{status}  '{text}'  →  {pred}")

✅  'Hello!'  →  greeting
✅  'Where are you?'  →  bot
✅  'Where is the Faculty of Engineering?'  →  location
✅  'What departments does NEU have?'  →  academic
✅  'Tell me about NEU research centres.'  →  general


## 14. Main pipeline: dataset doublecheck first, RAG + LLM fallback second

In [39]:
def format_sources(source_docs: list) -> str:
    """Convert retrieved Documents into a clean citation block."""
    seen  = set()
    lines = []
    for doc in source_docs:
        label = (
            doc.metadata.get("faculty_name")
            or doc.metadata.get("department_name")
            or doc.metadata.get("location_name")
            or doc.metadata.get("source", "NEU Knowledge Base")
        )
        # Clean up Windows paths -> keep only the filename
        if "\\" in label or "/" in label:
            label = label.replace("\\", "/").split("/")[-1]
            label = (label.replace("neu_", "")
                         .replace("_", " ")
                         .replace(".json", "")
                         .replace(".txt", "")
                         .replace(".md", "")
                         .title())
        if label not in seen:
            seen.add(label)
            lines.append(f"  \u2022 {label}")
    return "Sources retrieved:\n" + "\n".join(lines) if lines else ""


GREETING_RESPONSE = (
    "Hello! \U0001F44B Welcome to NEU Assistant. I'm here to help you with anything about "
    "Near East University \u2014 faculties, departments, campus facilities, research, "
    "admissions, and more. What would you like to know?"
)

BOT_RESPONSE = (
    "I'm NEU Assistant, an AI chatbot for Near East University (NEU) in Nicosia, Cyprus. "
    "I'm here to help you with questions about faculties, departments, admissions, "
    "campus facilities, research, and more. How can I help you?"
)

# Note: GREETING_PATTERNS is defined once, in step 13, and already includes
# "how are you" / "what's up" etc. so small local LLMs re-greeting on every
# answer still get caught by strip_greeting_leadin below.

GREETING_LEADIN_RE = re.compile(
    r"^\s*(hello|hi|hey)[!,.]?\s*(i'?m|i am)?\s*(neu assistant)?[^.\n]*[.!\n]\s*",
    re.IGNORECASE
)

def strip_greeting_leadin(answer: str) -> str:
    """Remove a redundant greeting/self-introduction sentence the LLM
    sometimes prepends, even on non-greeting questions."""
    cleaned = GREETING_LEADIN_RE.sub("", answer, count=1).strip()
    # Guard: if stripping wiped out the whole answer, keep the original.
    return cleaned if cleaned else answer


def ask(question: str, verbose: bool = True) -> dict:
    """
    Main pipeline, dataset-first (matches the architecture diagram):

      1. Greeting / bot-directed -> instant canned response (no retrieval, no LLM).
      2. DOUBLECHECK PASS (no LLM): retrieve top-K chunks from the vectorstore via
         dataset_answer() and check if the best one is similar enough to trust.
         If yes, that chunk's own text IS the final answer.
      3. LLM FALLBACK: only if step 2 found nothing confident enough, send the
         retrieved chunks + question to the LLM RAG chain (qa_chain).
    """
    question = correct_spelling(question)
    intent = detect_intent(question)

    if intent == "greeting":
        result = {"intent": intent, "answer": GREETING_RESPONSE, "sources": "", "method": "rule-based"}

    elif intent == "bot":
        result = {"intent": intent, "answer": BOT_RESPONSE, "sources": "", "method": "rule-based"}

    else:
        # --- STEP 2: doublecheck pass -- single retrieval + confidence check, zero LLM calls ---
        ds_hit = dataset_answer(question)

        if ds_hit is not None:
            result = {
                "intent": intent,
                "answer": ds_hit["answer"],
                "sources": f"Sources retrieved:\n  \u2022 {ds_hit['source']}",
                "method": ds_hit["method"],
            }
        else:
            # --- STEP 3: LLM fallback -- only reached if the dataset had no confident answer ---
            raw    = qa_chain.invoke({"query": question})
            answer = strip_greeting_leadin(raw["result"])
            sources = format_sources(raw["source_documents"])
            result = {
                "intent": intent,
                "answer": answer,
                "sources": sources,
                "method": "llm-fallback (no confident dataset match)",
            }

    if verbose:
        print(f"[{result['intent'].upper()}]  (method: {result['method']})")
        print(result["answer"])
        if result["sources"]:
            print("\n" + result["sources"])

    return result

print("ask() rewired: dataset-first (no LLM), LLM only as a fallback. \u2705")


ask() rewired: dataset-first (no LLM), LLM only as a fallback. ✅


## 15. Smoke tests

In [40]:
# Greeting
ask("Hello!")

# Academic
ask("What departments are in the Faculty of Engineering?")

# Location
ask("Where is the Grand Library on campus?")

# General
ask("What research centers does NEU have?")

[GREETING]  (method: rule-based)
Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, departments, campus facilities, research, admissions, and more. What would you like to know?
  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: dataset-match (chunk, score=0.92))
https://muhendislik.neu.edu.tr

Sources retrieved:
  • neu_qa_dataset.txt
[LOCATION]  (method: dataset-match (chunk, score=0.96))
The Grand Library is a standalone building in the central area of the NEU campus, open 24/7, covering 15,000 m². It contains 12 study rooms, 4 amphitheatres, a children's library, Book House (bookstore), American Corner, CMC archive, and Rare Works section. Director: Emre Karahasan, ext. 5536.

Sources retrieved:
  • neu_qa_dataset.txt
[GENERAL]  (method: dataset-match (chunk, score=0.99))
NEU has 28+ research centres, including: NEU-IBM Research Centre (with a supercomputer ranked 13th in the world), DESAM Research Institute, 

{'intent': 'general',
 'answer': 'NEU has 28+ research centres, including: NEU-IBM Research Centre (with a supercomputer ranked 13th in the world), DESAM Research Institute, Near East Institute, Experimental Animals Research Centre, Nuclear Medicine Research Centre, Herbarium (est. 2006), and 40+ state-of-the-art laboratories.',
 'sources': 'Sources retrieved:\n  • neu_qa_dataset.txt',
 'method': 'dataset-match (chunk, score=0.99)'}

In [41]:
test_questions = [
    "Hi there!",
    "Good morning!",
    "What departments are in the Faculty of Arts and Sciences?",
    "What degree types does the Faculty of Dentistry offer?",
    "What are the admission requirements at NEU?",
    "Where is the Grand Library?",
    "Where can I find the surgery room in the hospital?",
    "Does NEU have an Artificial Intelligence programme?",
    "What is the website of the Faculty of Dentistry?",
    "where is innovation bulding"
]

for q in test_questions:
    r = ask(q, verbose=False)
    preview = r['answer'][:120].replace("\n", " ")
    print(f"[{r['intent'].upper():<8}] {q}")
    print(f"           method \u2192 {r['method']}")
    print(f"           answer \u2192 {preview}...\n")


[GREETING] Hi there!
           method → rule-based
           answer → Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, department...

[GREETING] Good morning!
           method → rule-based
           answer → Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, department...

  [spell-check] 'departments' → 'department'
  [spell-check] 'Sciences?' → 'science'
[ACADEMIC] What departments are in the Faculty of Arts and Sciences?
           method → dataset-match (chunk, score=0.99)
           answer → Molecular Biology & Genetics, English Language and Literature, Mathematics, Turkish Language and Literature, Psychology ...

[ACADEMIC] What degree types does the Faculty of Dentistry offer?
           method → dataset-match (chunk, score=0.87)
           answer → https://dentistry.neu.edu.tr...

[ACADEMIC] What are the admission requirements at NEU?
           method → d

[LOCATION] Where is the Grand Library?
           method → dataset-match (chunk, score=0.96)
           answer → The Grand Library is a standalone building in the central area of the NEU campus, open 24/7, covering 15,000 m². It cont...

[LOCATION] Where can I find the surgery room in the hospital?
           method → dataset-match (chunk, score=0.83)
           answer → General Surgery outpatient clinic is on the 1st Floor, West Block, open Monday–Friday 08:00–17:00....

[ACADEMIC] Does NEU have an Artificial Intelligence programme?
           method → dataset-match (chunk, score=0.95)
           answer → Yes. The Faculty of Artificial Intelligence and Informatics is a dedicated faculty for AI programs, offering BSc, MSc, a...

[ACADEMIC] What is the website of the Faculty of Dentistry?
           method → dataset-match (chunk, score=1.00)
           answer → https://dentistry.neu.edu.tr...

  [spell-check] 'bulding' → 'building'
[LOCATION] where is innovation bulding
           metho

In [42]:
ask("Where is the Grand Library on campus?")

[LOCATION]  (method: dataset-match (chunk, score=0.96))
The Grand Library is a standalone building in the central area of the NEU campus, open 24/7, covering 15,000 m². It contains 12 study rooms, 4 amphitheatres, a children's library, Book House (bookstore), American Corner, CMC archive, and Rare Works section. Director: Emre Karahasan, ext. 5536.

Sources retrieved:
  • neu_qa_dataset.txt


{'intent': 'location',
 'answer': "The Grand Library is a standalone building in the central area of the NEU campus, open 24/7, covering 15,000 m². It contains 12 study rooms, 4 amphitheatres, a children's library, Book House (bookstore), American Corner, CMC archive, and Rare Works section. Director: Emre Karahasan, ext. 5536.",
 'sources': 'Sources retrieved:\n  • neu_qa_dataset.txt',
 'method': 'dataset-match (chunk, score=0.96)'}

In [43]:
ask("Room U08")

[LOCATION]  (method: exact-room-match (U08))
Room U08 ("Grand Library Archives") is located in the Innovation Building, level_u (underground floor).

Sources retrieved:
  • level_u.csv


{'intent': 'location',
 'answer': 'Room U08 ("Grand Library Archives") is located in the Innovation Building, level_u (underground floor).',
 'sources': 'Sources retrieved:\n  • level_u.csv',
 'method': 'exact-room-match (U08)'}

In [44]:
ask("Çagri Özkan, MSe")

[GENERAL]  (method: exact-lecturer-match (Çagri Özkan, MSe))
Çagri Özkan, MSe (Information Systems Engineering) is based in Room 109 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr. Rahib Abiyev (Computer Engineering - HoD), Assoc. Prof. Dr.John Bush (Cyber Security Engineering - HoD), Asst. Prof. Dr. Amr Abdelbari (Information Systems
Engineering-HoD).

Sources retrieved:
  • level_u.csv


{'intent': 'general',
 'answer': 'Çagri Özkan, MSe (Information Systems Engineering) is based in Room 109 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr. Rahib Abiyev (Computer Engineering - HoD), Assoc. Prof. Dr.John Bush (Cyber Security Engineering - HoD), Asst. Prof. Dr. Amr Abdelbari (Information Systems\nEngineering-HoD).',
 'sources': 'Sources retrieved:\n  • level_u.csv',
 'method': 'exact-lecturer-match (Çagri Özkan, MSe)'}

## 16. Flask backend

In [45]:
# ============================================================
# PASTE THIS AS A NEW CELL AT THE END OF YOUR NOTEBOOK
# (must run AFTER the cells that define: splitter, vectorstore,
#  Document, and ask() — i.e. after cell 20 / the RAG chain setup)
#
# Starts a Flask web server so the HTML chatbot UI *and* the new
# admin page can:
#   - send questions to your ask() function
#   - add new documents straight into your FAISS vectorstore
#   - list what's currently indexed
#   - delete a document's chunks from the vectorstore
#   - update (replace) a document's chunks in place
# ============================================================

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import os
import uuid
from datetime import datetime

app = Flask(__name__)
CORS(app)  # allows your frontend to call it from a different port

# ------------------------------------------------------------
# Where uploaded files get saved on disk, and a running record of
# what's been added to the vectorstore this session.
#
# Each item now also stores the exact vectorstore chunk `ids` that
# came back from `vectorstore.add_documents(...)`, so we can later
# delete or replace *just* that document's chunks without touching
# anything else in the index.
# ------------------------------------------------------------
UPLOAD_DIR = "uploaded_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)

ALLOWED_EXTENSIONS = {".txt", ".pdf", ".docx", ".csv", ".md"}

# Each item: {"id", "filename", "chunks_added", "ids": [...], "uploaded_at"}
uploaded_documents = []


def extract_text(filepath, ext):
    """Pull plain text out of the uploaded file based on its extension."""
    if ext in (".txt", ".md", ".csv"):
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    if ext == ".pdf":
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader  # fallback if pypdf isn't installed
        reader = PdfReader(filepath)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    if ext == ".docx":
        import docx
        doc = docx.Document(filepath)
        return "\n".join(p.text for p in doc.paragraphs)

    return ""


def find_document(doc_id):
    return next((d for d in uploaded_documents if d["id"] == doc_id), None)


def add_document_to_index(text, filename):
    """
    Wraps the uploaded text in a LangChain Document, splits it with the
    SAME splitter used to build the original knowledge base, and adds
    the resulting chunks directly into the existing `vectorstore`.

    Because `retriever` and `dataset_answer()` both read from this same
    `vectorstore` object (not a copy), new content becomes searchable by
    both the RAG/LLM path and the no-LLM doublecheck path immediately —
    no restart, no rebuild.

    Returns (chunk_ids, num_chunks) so the caller can track what was
    added and delete/replace it later.
    """
    new_doc = Document(
        page_content=text,
        metadata={"source": filename, "type": "uploaded"},
    )
    new_chunks = splitter.split_documents([new_doc])
    chunk_ids = vectorstore.add_documents(new_chunks)

    print(f"📄 Added '{filename}' to vectorstore as {len(new_chunks)} chunk(s).")
    return chunk_ids, len(new_chunks)


def remove_chunks_from_index(chunk_ids):
    """Delete a specific set of chunk ids from the vectorstore, if any exist."""
    if chunk_ids:
        vectorstore.delete(ids=chunk_ids)


@app.route("/ask", methods=["POST"])
def ask_endpoint():
    data = request.get_json()
    question = data.get("question", "").strip()
    if not question:
        return jsonify({"error": "No question provided"}), 400
    try:
        result = ask(question, verbose=False)
        return jsonify({
            "answer": result["answer"],
            "intent": result["intent"],
            "sources": result["sources"]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/upload", methods=["POST"])
def upload_endpoint():
    if "file" not in request.files:
        return jsonify({"error": "No file included in the request."}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400

    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        return jsonify({
            "error": f"Unsupported file type '{ext}'. "
                     f"Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}"
        }), 400

    save_path = os.path.join(UPLOAD_DIR, file.filename)
    file.save(save_path)

    try:
        text = extract_text(save_path, ext)
    except Exception as e:
        return jsonify({"error": f"Could not read file: {e}"}), 500

    if not text.strip():
        return jsonify({"error": "No readable text found in file."}), 400

    try:
        chunk_ids, chunks_added = add_document_to_index(text, file.filename)
    except Exception as e:
        return jsonify({"error": f"Could not index file: {e}"}), 500

    entry = {
        "id": str(uuid.uuid4()),
        "filename": file.filename,
        "chunks_added": chunks_added,
        "ids": chunk_ids,
        "uploaded_at": datetime.utcnow().isoformat() + "Z",
    }
    uploaded_documents.append(entry)

    return jsonify({
        "message": f"'{file.filename}' uploaded and added to the knowledge base.",
        "document": {k: v for k, v in entry.items() if k != "ids"},
    })


@app.route("/documents", methods=["GET"])
def list_documents():
    """See what's been uploaded (and indexed) this session."""
    # Don't leak raw vectorstore chunk ids to the frontend — not needed there.
    docs = [{k: v for k, v in d.items() if k != "ids"} for d in uploaded_documents]
    return jsonify({"documents": docs})


@app.route("/documents/<doc_id>", methods=["DELETE"])
def delete_document(doc_id):
    """Remove a document's chunks from the vectorstore and drop its file."""
    entry = find_document(doc_id)
    if not entry:
        return jsonify({"error": "Document not found."}), 404

    try:
        remove_chunks_from_index(entry["ids"])
    except Exception as e:
        return jsonify({"error": f"Could not remove from vectorstore: {e}"}), 500

    save_path = os.path.join(UPLOAD_DIR, entry["filename"])
    if os.path.exists(save_path):
        os.remove(save_path)

    uploaded_documents.remove(entry)
    return jsonify({"message": f"'{entry['filename']}' removed from the knowledge base."})


@app.route("/documents/<doc_id>", methods=["PUT"])
def update_document(doc_id):
    """
    Replace a document with a new version: delete its old chunks from the
    vectorstore, then index the newly uploaded file in its place.
    """
    entry = find_document(doc_id)
    if not entry:
        return jsonify({"error": "Document not found."}), 404

    if "file" not in request.files:
        return jsonify({"error": "No replacement file included in the request."}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400

    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        return jsonify({
            "error": f"Unsupported file type '{ext}'. "
                     f"Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}"
        }), 400

    save_path = os.path.join(UPLOAD_DIR, file.filename)
    file.save(save_path)

    try:
        text = extract_text(save_path, ext)
    except Exception as e:
        return jsonify({"error": f"Could not read file: {e}"}), 500

    if not text.strip():
        return jsonify({"error": "No readable text found in file."}), 400

    try:
        new_ids, new_count = add_document_to_index(text, file.filename)
        remove_chunks_from_index(entry["ids"])  # remove old only after new succeeds
    except Exception as e:
        return jsonify({"error": f"Could not index replacement file: {e}"}), 500

    # Clean up old saved file if the filename changed
    old_path = os.path.join(UPLOAD_DIR, entry["filename"])
    if entry["filename"] != file.filename and os.path.exists(old_path):
        os.remove(old_path)

    entry["filename"] = file.filename
    entry["chunks_added"] = new_count
    entry["ids"] = new_ids
    entry["uploaded_at"] = datetime.utcnow().isoformat() + "Z"

    return jsonify({
        "message": f"'{file.filename}' replaced the previous version in the knowledge base.",
        "document": {k: v for k, v in entry.items() if k != "ids"},
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

# Run in a background thread so Jupyter doesn't freeze
thread = threading.Thread(target=lambda: app.run(port=5050, debug=False, use_reloader=False))
thread.daemon = True
thread.start()
print("Flask server running on http://localhost:5050 ✅")
print("Chat UI hits /ask. Admin page hits /documents (GET/DELETE/PUT) and /upload (POST).")


Flask server running on http://localhost:5050 ✅
Chat UI hits /ask. Admin page hits /documents (GET/DELETE/PUT) and /upload (POST).
 * Serving Flask app '__main__'


 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit
